In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
from transformers import pipeline,AutoTokenizer,AutoModelForSequenceClassification
import torch
warnings.filterwarnings("ignore")

In [2]:
input_file="2. Reviews.csv"
output_folder="4. Outputs"
model="cardiffnlp/twitter-roberta-base-sentiment-latest"
batch_size=16
max_tokens=512
sentiment_weightage=0.55
rating_weightage=0.45
bootstrap=1000
random_seed=42

In [3]:
np.random.seed(random_seed)
os.makedirs(output_folder,exist_ok=True)

In [4]:
df=pd.read_csv(input_file)

In [5]:
df.head(20)

,Mobile_Device,Rating,Title,Review,Reviewer_Name,Date,Additional_Information
0,iPhone 17 Pro Max,5,Iphone 17 pro max,The iPhone 17 Pro Max offers a large 6.9-inch ...,Chakri,5 months ago,Incentivized | Verified Purchaser
1,iPhone 17 Pro Max,5,iPhone 17 pro max,"the iPhone 17 Pro, a marvel in tech's garden! ...",MirianA,1 month ago,Incentivized | Verified Purchaser | Owned for ...
2,iPhone 17 Pro Max,5,Absolutely worth the upgrade,This phone exceeded my expectations. The perfo...,RayssaC,2 months ago,Incentivized | Verified Purchaser | Owned for ...
3,iPhone 17 Pro Max,5,Love it,"Absolutely love this phone, coming from a 14 p...",Summer,4 months ago,Incentivized | Verified Purchaser | Owned for ...
4,iPhone 17 Pro Max,4,Apples best but not perfect,What else can be said about this product other...,MarioN,5 months ago,Incentivized | Verified Purchaser | Best Buy E...
5,iPhone 17 Pro Max,5,iphone 17 pro max silver,Love my iphone 17! Good battery. love the came...,Amyafontano,3 months ago,Incentivized | Verified Purchaser | Owned for ...
6,iPhone 17 Pro Max,5,Excellent,The iPhone 17 Pro Max is an insanely good phon...,Felico,5 months ago,Incentivized | Verified Purchaser
7,iPhone 17 Pro Max,5,iPhone 17 pro max,The duration of battery is the best and the ca...,user568040,5 months ago,Incentivized | Verified Purchaser
8,iPhone 17 Pro Max,5,17 pro max,I love it. The camera is amazing and the batte...,Matthew,5 months ago,Incentivized | Verified Purchaser
9,iPhone 17 Pro Max,5,best phone for content creation,im so glad i upgraded it. I traded in my 15pro...,KeniaV,2 months ago,Incentivized | Verified Purchaser | Owned for ...


In [6]:
print(df.isnull().sum())

Mobile_Device              0
Rating                     0
Title                      0
Review                     0
Reviewer_Name             15
Date                      15
Additional_Information     0
dtype: int64


In [7]:
def preprocessing(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text=text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^\w\s.,!?']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [8]:
df["review_clean"]=df["Review"].apply(preprocessing)
df["title_clean"]=df["Title"].apply(preprocessing)

In [9]:
df["review_clean"]

0       the iphone 17 pro max offers a large 6.9 inch ...
1       the iphone 17 pro, a marvel in tech's garden! ...
2       this phone exceeded my expectations. the perfo...
3       absolutely love this phone, coming from a 14 p...
4       what else can be said about this product other...
                              ...                        
1118    all good. does what is expected. just purchase...
1119    great phone, the camera is awesome. so far i a...
1120    they didn't deliver it on time, and they gave ...
1121    it has been 2 weeks since they ssaid the phone...
1122    i haven't received my order it's already been ...
Name: review_clean, Length: 1123, dtype: object

In [10]:
df["title_clean"]

0                      iphone 17 pro max
1                      iphone 17 pro max
2           absolutely worth the upgrade
3                                love it
4            apples best but not perfect
                      ...               
1118                     all good so far
1119                          best phone
1120                    terrible service
1121                          outrageous
1122    what is happening with my order?
Name: title_clean, Length: 1123, dtype: object

In [11]:
df["title_review"]=df["title_clean"]+"."+df["review_clean"]

In [12]:
df["title_review"]

0       iphone 17 pro max.the iphone 17 pro max offers...
1       iphone 17 pro max.the iphone 17 pro, a marvel ...
2       absolutely worth the upgrade.this phone exceed...
3       love it.absolutely love this phone, coming fro...
4       apples best but not perfect.what else can be s...
                              ...                        
1118    all good so far.all good. does what is expecte...
1119    best phone.great phone, the camera is awesome....
1120    terrible service.they didn't deliver it on tim...
1121    outrageous.it has been 2 weeks since they ssai...
1122    what is happening with my order?.i haven't rec...
Name: title_review, Length: 1123, dtype: object

In [13]:
df["rating_norm"]=(df["Rating"]-1)/4.0

In [14]:
df["review_len"]=df["review_clean"].str.len()

In [15]:
df["reviews_weight"]=np.log1p(df["review_len"])

In [16]:
print(f"\n{df['Mobile_Device'].value_counts().to_string()}")


Mobile_Device
Samsung Galaxy S26 Ultra    462
iPhone 17 Pro Max           331
Google Pixel 10 Pro         330


In [17]:
tokenizer=AutoTokenizer.from_pretrained(model)
modell=AutoModelForSequenceClassification.from_pretrained(model)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [18]:
pipe=pipeline(
    "text-classification",
    model=modell,
    tokenizer=tokenizer,
    truncation=True,
    max_length=max_tokens,
    return_all_scores=True,
    batch_size=batch_size,
)

Device set to use cpu


In [19]:
texts=df["title_review"].astype(str).tolist()

In [20]:
raw_results=[]

In [21]:
for i in range(0,len(texts),batch_size):
    batch=texts[i:i+batch_size]
    raw_results.extend(pipe(batch))

In [22]:
def sentiment(result_list: list) -> tuple:
    scores = {item["label"].lower(): item["score"] for item in result_list}
    pos=scores.get("positive",0)
    neu=scores.get("neutral",0)
    neg=scores.get("negative",0)
    sentiment_score = pos*1.0 + neu*0.5 + neg* 0.0
    all_scores= {"positive":pos, "neutral": neu, "negative": neg}
    dominant=max(all_scores,key=all_scores.get)
    return round(sentiment_score,4),dominant,round(pos,4),round(neu,4),round(neg,4)

In [23]:
parsed=[sentiment(r) for r in raw_results]

In [24]:
print(f"Total rows in DataFrame: {len(df)}")
print(f"Total items in TEXTS list: {len(texts)}")
print(f"Total results in PARSED list: {len(parsed)}")

Total rows in DataFrame: 1123
Total items in TEXTS list: 1123
Total results in PARSED list: 1123


In [25]:
df["sentiment_score"],df["sentiment_label"],df["positive_probability"],df["neutral_probability"],df["negative_probability"]=zip(*parsed)

In [26]:
print(df["sentiment_label"].value_counts().to_string())

sentiment_label
positive    1038
negative      64
neutral       21


In [27]:
df["satisfaction_score_raw"]=(
    sentiment_weightage * df["sentiment_score"] + rating_weightage * df["rating_norm"]
)

In [28]:
df["satisfaction_score_raw"]

0       0.986305
1       0.991805
2       0.995820
3       0.995820
4       0.854335
          ...   
1118    0.987460
1119    0.995545
1120    0.014245
1121    0.038720
1122    0.035035
Name: satisfaction_score_raw, Length: 1123, dtype: float64

In [29]:
scaler=MinMaxScaler(feature_range=(0,100))

In [30]:
df["satisfaction_score"]=scaler.fit_transform(df[["satisfaction_score_raw"]]).round(2)

In [31]:
df["satisfaction_score"]

0       98.91
1       99.47
2       99.88
3       99.88
4       85.49
        ...  
1118    99.03
1119    99.85
1120     0.00
1121     2.49
1122     2.12
Name: satisfaction_score, Length: 1123, dtype: float64

In [32]:
#checking the transformers result is valid as compared to the rating
df["rating_positive"]=df["Rating"]>=4
df["sentiment_positive"]=df["sentiment_label"]=="positive"
df["score_agreement"]=(df["rating_positive"] == df["sentiment_positive"]).astype(int)

In [33]:
print(f"\n {df['score_agreement'].mean()*100:.1f}% of reviews shows that the transformer results are correct as compared to the ratings")


 96.5% of reviews shows that the transformer results are correct as compared to the ratings


In [34]:
#Bootstrapping to handle imbalances in the dataset
devices=df["Mobile_Device"].unique()
min_count=df["Mobile_Device"].value_counts().min()

In [35]:
print(min_count)

330


In [36]:
bootstrap_res=[]
for d in devices:
    device_scores=df[df["Mobile_Device"]==d]["satisfaction_score"].values
    n=len(device_scores)
    boot_means=np.array([
        np.random.choice(device_scores,size=min_count,replace=True).mean()
    for _ in range(bootstrap)
    ])
    mean_score=boot_means.mean()
    ci_lower=np.percentile(boot_means,2.5)
    ci_upper=np.percentile(boot_means,97.5)
    std_error=boot_means.std()
    bootstrap_res.append({
       "Mobile_Device" :d,
        "Number Of Reviews" :n,
        "Bootstrap_Mean_Score" :round(mean_score,2),
        "ci_lower_95" :round(ci_lower,2),
        "ci_upper_95" :round(ci_upper,2),
        "std_error" :round(std_error,4),
        "raw_mean_score" :round(device_scores.mean(),2),
        "bias correction" :round(mean_score-device_scores.mean(),4),
    })
    print(f"  {d:<30} → Score: {mean_score:.2f}  95%CI [{ci_lower:.2f}, {ci_upper:.2f}]  (n={n})")

  iPhone 17 Pro Max              → Score: 94.31  95%CI [92.01, 96.13]  (n=331)
  Google Pixel 10 Pro            → Score: 92.85  95%CI [90.64, 94.76]  (n=330)
  Samsung Galaxy S26 Ultra       → Score: 92.24  95%CI [89.97, 94.38]  (n=462)


In [37]:
bootstrap_df=pd.DataFrame(bootstrap_res).sort_values("Bootstrap_Mean_Score",ascending=False).reset_index(drop=True)

In [38]:
bootstrap_df

,Mobile_Device,Number Of Reviews,Bootstrap_Mean_Score,ci_lower_95,ci_upper_95,std_error,raw_mean_score,bias correction
0,iPhone 17 Pro Max,331,94.31,92.01,96.13,1.0324,94.27,0.0382
1,Google Pixel 10 Pro,330,92.85,90.64,94.76,1.0766,92.80,0.0455
2,Samsung Galaxy S26 Ultra,462,92.24,89.97,94.38,1.1389,92.22,0.0274


In [39]:
Winner=bootstrap_df.iloc[0]["Mobile_Device"]

In [40]:
print(f"\n THE MOBILE DEVICE HAVING THE BEST CUSTOMER SATISFACTION RESULT IS {Winner}")


 THE MOBILE DEVICE HAVING THE BEST CUSTOMER SATISFACTION RESULT IS iPhone 17 Pro Max


In [41]:
df.head()

,Mobile_Device,Rating,Title,Review,Reviewer_Name,Date,Additional_Information,review_clean,title_clean,title_review,...,sentiment_score,sentiment_label,positive_probability,neutral_probability,negative_probability,satisfaction_score_raw,satisfaction_score,rating_positive,sentiment_positive,score_agreement
0,iPhone 17 Pro Max,5,Iphone 17 pro max,The iPhone 17 Pro Max offers a large 6.9-inch ...,Chakri,5 months ago,Incentivized | Verified Purchaser,the iphone 17 pro max offers a large 6.9 inch ...,iphone 17 pro max,iphone 17 pro max.the iphone 17 pro max offers...,...,0.9751,positive,0.9536,0.0430,0.0034,0.986305,98.91,True,True,1
1,iPhone 17 Pro Max,5,iPhone 17 pro max,"the iPhone 17 Pro, a marvel in tech's garden! ...",MirianA,1 month ago,Incentivized | Verified Purchaser | Owned for ...,"the iphone 17 pro, a marvel in tech's garden! ...",iphone 17 pro max,"iphone 17 pro max.the iphone 17 pro, a marvel ...",...,0.9851,positive,0.9730,0.0241,0.0029,0.991805,99.47,True,True,1
2,iPhone 17 Pro Max,5,Absolutely worth the upgrade,This phone exceeded my expectations. The perfo...,RayssaC,2 months ago,Incentivized | Verified Purchaser | Owned for ...,this phone exceeded my expectations. the perfo...,absolutely worth the upgrade,absolutely worth the upgrade.this phone exceed...,...,0.9924,positive,0.9888,0.0072,0.0040,0.995820,99.88,True,True,1
3,iPhone 17 Pro Max,5,Love it,"Absolutely love this phone, coming from a 14 p...",Summer,4 months ago,Incentivized | Verified Purchaser | Owned for ...,"absolutely love this phone, coming from a 14 p...",love it,"love it.absolutely love this phone, coming fro...",...,0.9924,positive,0.9890,0.0068,0.0041,0.995820,99.88,True,True,1
4,iPhone 17 Pro Max,4,Apples best but not perfect,What else can be said about this product other...,MarioN,5 months ago,Incentivized | Verified Purchaser | Best Buy E...,what else can be said about this product other...,apples best but not perfect,apples best but not perfect.what else can be s...,...,0.9397,positive,0.8985,0.0823,0.0192,0.854335,85.49,True,True,1


In [42]:
cols=[
    "Mobile_Device","Rating","sentiment_label","sentiment_score","positive_probability","neutral_probability","negative_probability","rating_norm","satisfaction_score","score_agreement","review_len","Date","Additional_Information"
]
df[cols].to_csv(f"{output_folder}/01_reviews_full.csv",index=False)

In [43]:
device_summary=df.groupby("Mobile_Device").agg(
    n_reviews=("satisfaction_score","count"),
    mean_satisfaction=("satisfaction_score","mean"),
    median_satisfaction=("satisfaction_score","median"),
    std_satisfaction=("satisfaction_score","std"),
    mean_rating=("Rating","mean"),
    mean_sentiment_score=("sentiment_score","mean"),
    Percentage_Positive=("sentiment_positive","mean"),
    Percentage_Agreement=("score_agreement","mean"),
    Percentage_5Star=("Rating", lambda x: (x==5).mean()),
    Percentage_1Star=("Rating", lambda x: (x==1).mean()),
).reset_index()

In [44]:
for c in ["Percentage_Positive", "Percentage_Agreement", "Percentage_5Star", "Percentage_1Star"]:
    device_summary[c]=(device_summary[c]*100).round(2)
for c in ["mean_satisfaction","median_satisfaction","std_satisfaction","mean_rating","mean_sentiment_score"]:
    device_summary[c]=device_summary[c].round(2)

In [45]:
device_summary=device_summary.merge(
    bootstrap_df[["Mobile_Device","Bootstrap_Mean_Score","ci_upper_95","ci_upper_95","std_error","bias correction"]],
    on="Mobile_Device"
)

In [46]:
device_summary["satisfaction_rank"]=device_summary["Bootstrap_Mean_Score"].rank(ascending=False).astype(int)

In [47]:
device_summary.to_csv(f"{output_folder}/02_device_summary.csv",index=False)

In [48]:
rating_distribution=df.groupby(["Mobile_Device","Rating"]).agg(
    count=("satisfaction_score","count"),
    avg_sentiment_score=("sentiment_score","mean"),
    avg_satisfaction=("satisfaction_score","mean"),
).reset_index()

In [49]:
rating_distribution["Percentage_of_device"]=rating_distribution.groupby("Mobile_Device")["count"].transform(
    lambda x: (x/x.sum()*100).round(2)
)

In [50]:
rating_distribution.to_csv(f"{output_folder}/03_rating_distribution.csv",index=False)

In [51]:
sentiment_distribution=df.groupby(["Mobile_Device","sentiment_label"]).agg(
    count=("satisfaction_score","count"),
    avg_satisfaction=("satisfaction_score","mean"),
    avg_rating=("Rating","mean"),
).reset_index()

In [52]:
sentiment_distribution["Percentage_of_device"]=sentiment_distribution.groupby("Mobile_Device")["count"].transform(
    lambda x:(x/x.sum()*100).round(2)
)

In [53]:
sentiment_distribution.to_csv(f"{output_folder}/04_sentiment_distribution.csv",index=False)

In [54]:
agreement_df=df.groupby(["Mobile_Device","score_agreement"]).agg(
    count=("satisfaction_score","count"),
    avg_satisfaction=("satisfaction_score","mean"),
).reset_index()

In [55]:
agreement_df["agreement_label"]=agreement_df["score_agreement"].map(
    {1: "Agree", 0: "Disagree"}
)

In [56]:
agreement_df["Percentage_of_device"]=agreement_df.groupby("Mobile_Device")["count"].transform(
    lambda x:(x/x.sum()*100).round(2)
)

In [57]:
agreement_df.to_csv(f"{output_folder}/05_score_agreement.csv",index=False)

In [58]:
boot_dist_rows=[]
for d in devices:
    dev_scores=df[df["Mobile_Device"]==d]["satisfaction_score"].values
    for i in range(bootstrap):
        sample_mean=np.random.choice(dev_scores,size=min_count,replace=True).mean()
        boot_dist_rows.append({"Mobile_Device": d, "bootstrap_sample_mean": round(sample_mean,4)})

In [59]:
boot_dist_df=pd.DataFrame(boot_dist_rows)
boot_dist_df.to_csv(f"{output_folder}/06_bootstrap_distribution.csv",index=False)

In [60]:
def score_band(score):
    if score>=80: return "Excellent(80-100)"
    elif score>=60: return "Good(60-79)"
    elif score>=40: return "Average(40-59)"
    else:
        return "Poor(0-39)"

In [61]:
df["score_band"]=df["satisfaction_score"].apply(score_band)

In [62]:
band_distribution=df.groupby(["Mobile_Device","score_band"]).agg(
    count=("satisfaction_score","count"),
    avg_satisfaction=("satisfaction_score","mean"),
).reset_index()

In [63]:
band_distribution["Percentage_of_device"]=band_distribution.groupby("Mobile_Device")["count"].transform(
    lambda x:(x/x.sum()*100).round(2)
)

In [64]:
band_order=["Excellent(80-100)","Good(60-79)","Average(40-59)","Poor(0-39)"]
band_distribution["band_order"]=band_distribution["score_band"].map({v: i for i,v in enumerate(band_order)})
band_distribution=band_distribution.sort_values(["Mobile_Device","band_order"])

In [65]:
band_distribution.drop("band_order",axis=1).to_csv(f"{output_folder}/07_score_bands.csv",index=False)

In [66]:
print(f"\n Bootstrap params :: N={bootstrap}, sample_size={min_count}")
print(f"\n Score weights :: Sentiment={sentiment_weightage} | Rating={rating_weightage}")
print(f"\n Transformer model = {model}")


 Bootstrap params :: N=1000, sample_size=330

 Score weights :: Sentiment=0.55 | Rating=0.45

 Transformer model = cardiffnlp/twitter-roberta-base-sentiment-latest


In [67]:
print(device_summary[[
    "Mobile_Device", "n_reviews", "satisfaction_rank",
    "Percentage_Positive", "mean_rating"
]].to_string(index=False))

           Mobile_Device  n_reviews  satisfaction_rank  Percentage_Positive  mean_rating
     Google Pixel 10 Pro        330                  2                92.73         4.73
Samsung Galaxy S26 Ultra        462                  3                91.13         4.74
       iPhone 17 Pro Max        331                  1                93.96         4.80


In [68]:
print(f"\n WINNER  = {Winner}")


 WINNER  = iPhone 17 Pro Max
